# Ambig — Ambiguity-code robustness, all five models, CD-HIT-97%, ORIGINAL CNN architectures

**Why this notebook exists.** T12 measured ambiguity-injection robustness but only ever ran the
linear SVM and random forest, on the MinHash split. Your Abstract and Conclusion currently claim
the deep networks "degraded substantially more" under ambiguity -- a claim with no CNN data
behind it. This notebook fixes that: linear SVM, random forest, and all three CNNs, on the
CD-HIT-97% split, using the exact original CNN architectures and training recipe from your
three per-architecture notebooks (not the simplified versions from the old Fragsweep) -- see the note
in Fragsweep for the full list of differences this corrects (GaussianNoise, graduated stochastic
depth, the real Inception reduction branches, VGG's extra dense layer, Adam(3e-4), 120 epochs
with ReduceLROnPlateau, per-feature StandardScaler, one-hot CategoricalCrossentropy).

**GPU needed.** Yes. **Budget heavily** -- with the original (heavier) architectures, each CNN
fit is roughly 15-30 minutes, not the 2-4 minutes the simplified architecture took. A grid of
(ambiguity rates) x (lengths) x (seeds) x (3 CNNs) gets expensive fast.

**Runtime and the reduced default grid.** `FAST_MODE = True` (default) runs 2 ambiguity rates
([0.0, 0.10]) x 2 lengths ([300, full]) x 2 seeds x 3 CNNs = 24 CNN fits, roughly 6-12 hours --
still likely more than one Kaggle session. This notebook is **resume-safe**: it loads any
existing `ambig_ambiguity_all5_raw.csv` and skips (model, length, ambiguity_rate, seed) combinations
already computed, so you can stop and restart across sessions without losing work or wasting
quota. Expand `AMBIG_RATES` / `LENGTHS` / `SEEDS` once you've confirmed the reduced grid runs
cleanly and you have GPU quota to spare. The classical half (SVM/RF) is CPU-only and fast
regardless of grid size.

**Dataset.** All notebooks below read the *same* dataset you already used for the main experiments and the 5-model fragment sweep:

`/kaggle/input/datasets/yashrajrajarishi/subtype-classification/prepared_sequences_clean.csv`

with columns `sequence`, `subtype`, `cluster` (the MinHash cluster id). Attach that Kaggle Dataset to the notebook the same way you did for the fragment-test notebook (Add Data -> Your Datasets -> subtype-classification). **No new download is needed** for this file. Where a test needs something extra (a reference genome, an external tool), that is called out explicitly at the top of the notebook, with the exact cell that fetches or installs it -- it downloads automatically inside the notebook, you do not need to upload anything yourself.


**Kaggle session budget.** Kaggle notebook sessions cap out at **12 hours** and (for GPU) a **~30 GPU-hours/week** quota. Each notebook below states whether it needs a GPU and gives a rough runtime estimate at the default settings, plus a `FAST_MODE` toggle to shrink seeds/epochs/lengths if you are short on quota. If a run is approaching the 12 h wall, results are saved to `/kaggle/working` incrementally (per seed / per length) so a partial run is still usable -- just re-run with the remaining settings and concatenate the CSVs.


**Editorial note (clean repo build).** The `build_resnet18` cell below has had its `GaussianNoise(0.05)` input layer removed, matching the final no-noise architecture used throughout this study. SVM, RF, VGG-16 and Inception-v1 were never affected. `ambiguity_all5_classical_vgg_inception_and_legacy_resnet_raw.csv` (this notebook's own output) still contains the old, superseded GaussianNoise-era ResNet-18 rows alongside the unaffected SVM/RF/VGG-16/Inception-v1 rows; the authoritative no-noise ResNet-18 numbers are in `resnet18_ambiguity_nonoise_raw.csv` / `_summary.csv`. GaussianNoise removed from 1 cell(s).

### 1. Imports and install CD-HIT

In [ ]:

import warnings; warnings.filterwarnings("ignore")
import os, subprocess, shutil
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, balanced_accuracy_score

from tensorflow.keras.layers import (Input, Conv1D, BatchNormalization, ReLU, Add,
    GlobalAveragePooling1D, Dropout, Dense, MaxPooling1D, GaussianNoise,
    SpatialDropout1D, Layer, Concatenate)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical

OUT = "/kaggle/working"
os.makedirs(OUT, exist_ok=True)

if shutil.which("cd-hit-est") is None:
    print("cd-hit-est not found, installing...")
    subprocess.run("conda install -y -c bioconda cd-hit 2>/dev/null || true", shell=True)
    if shutil.which("cd-hit-est") is None:
        subprocess.run(
            "wget -q https://github.com/weizhongli/cdhit/releases/download/V4.8.1/cd-hit-v4.8.1-2019-0228.tar.gz "
            "&& tar xzf cd-hit-v4.8.1-2019-0228.tar.gz && cd cd-hit-v4.8.1-2019-0228 && make -j4 "
            "&& cp cd-hit-est /usr/local/bin/ && cp cd-hit /usr/local/bin/", shell=True)
print("cd-hit-est available:", shutil.which("cd-hit-est") is not None)


### 2. Config

In [ ]:

FAST_MODE = False

AMBIG_RATES = [0.0, 0.10] if FAST_MODE else [0.0, 0.005, 0.01, 0.02, 0.05, 0.10]
LENGTHS     = [300, None] if FAST_MODE else [300, 500, 1000, None]   # None = full length
SEEDS       = [0, 1] if FAST_MODE else [0, 1, 2]
DEEP_MODELS = ["resnet18", "vgg16", "inception"]
MAX_TFIDF_FEATS = 512
IDENTITY = 0.97
EPOCHS = 120
BATCH  = 48


### 3. Load data and build the CD-HIT-97% grouping

In [ ]:

DATA_PATH = "/kaggle/input/datasets/yashrajrajarishi/subtype-classification/prepared_sequences_clean.csv"
df = pd.read_csv(DATA_PATH).reset_index(drop=True)
le = LabelEncoder(); y_all = le.fit_transform(df["subtype"].values)
CLASSES = list(le.classes_); n_cls = len(CLASSES)
X_all = df["sequence"].values
n = len(df)
print(f"{n} sequences, {n_cls} classes: {CLASSES}")

with open(f"{OUT}/pol_ambig.fasta", "w") as f:
    for i, s in enumerate(X_all):
        f.write(f">{i}\n{s}\n")

cmd = f"cd-hit-est -i {OUT}/pol_ambig.fasta -o {OUT}/pol_ambig_cdhit97 -c {IDENTITY} -n 10 -M 16000 -T 4 -d 0"
r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
assert r.returncode == 0, r.stderr

def parse_cdhit_clusters(clstr_path, n_seqs):
    labels = np.full(n_seqs, -1, dtype=int)
    cid = -1
    with open(clstr_path) as f:
        for line in f:
            if line.startswith(">Cluster"):
                cid += 1
            else:
                idx = int(line.split(">")[1].split("...")[0])
                labels[idx] = cid
    assert (labels >= 0).all()
    return labels

grp_all = parse_cdhit_clusters(f"{OUT}/pol_ambig_cdhit97.clstr", n)
print(f"CD-HIT 97% identity: {len(set(grp_all))} clusters")


### 4. Ambiguity injector (identical to T12), fragment/split helpers

In [ ]:

AMBIG = list("RYKMSWBDHVN")

def inject_ambiguity(seq, rate, rng):
    if rate <= 0:
        return seq
    s = list(seq)
    n_to_change = int(len(s) * rate)
    if n_to_change == 0:
        return seq
    idx = rng.choice(len(s), size=n_to_change, replace=False)
    for i in idx:
        s[i] = rng.choice(AMBIG)
    return "".join(s)

def fragment(seq, length, rng):
    if length is None or len(seq) <= length:
        return seq
    start = rng.integers(0, len(seq) - length + 1)
    return seq[start:start+length]

def split_grouped(X, y, grp, seed):
    s1 = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
    trv, te = next(s1.split(X, y, grp))
    s2 = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=seed)
    tr_rel, va_rel = next(s2.split(X[trv], y[trv], grp[trv]))
    return trv[tr_rel], trv[va_rel], te

def fit_tfidf(seq_tr, seq_va, seq_te):
    vec = TfidfVectorizer(analyzer="char", ngram_range=(6,6), max_features=MAX_TFIDF_FEATS, lowercase=False)
    Xtr = vec.fit_transform(seq_tr); Xva = vec.transform(seq_va); Xte = vec.transform(seq_te)
    return Xtr, Xva, Xte


### 5. The ORIGINAL CNN architectures (same as Fragsweep -- copied verbatim, do not simplify)

In [ ]:

class StochasticDepth(Layer):
    def __init__(self, p=0.85, **kw): super().__init__(**kw); self.p = p
    def call(self, x, training=None):
        if not training: return x
        r = tf.random.uniform([tf.shape(x)[0], 1, 1])
        return x * tf.floor(self.p + r) / self.p
    def get_config(self): return {**super().get_config(), "p": self.p}

def res_block(x, f, stride=1, sp=0.85, dr=0.1):
    sc = x
    x = Conv1D(f, 3, strides=stride, padding="same", use_bias=False)(x)
    x = ReLU()(BatchNormalization()(x)); x = SpatialDropout1D(dr)(x)
    x = BatchNormalization()(Conv1D(f, 3, padding="same", use_bias=False)(x))
    if stride != 1 or sc.shape[-1] != f:
        sc = BatchNormalization()(Conv1D(f, 1, strides=stride, padding="same", use_bias=False)(sc))
    return ReLU()(Add()([StochasticDepth(sp)(x), sc]))

def build_resnet18(input_len, n_classes):
    inp = Input((input_len, 1)); x = inp  # GaussianNoise REMOVED -- final architecture used throughout this study has no input-noise augmentation
    x = ReLU()(BatchNormalization()(Conv1D(32, 7, strides=2, padding="same", use_bias=False)(x)))
    x = MaxPooling1D(3, strides=2, padding="same")(x)
    for f, sp in [(32, .95), (32, .92)]:  x = res_block(x, f, 1, sp)
    for f, sp in [(64, .88), (64, .85)]:  x = res_block(x, f, 2 if f != x.shape[-1] else 1, sp)
    for f, sp in [(128, .80), (128, .78)]: x = res_block(x, f, 2 if f != x.shape[-1] else 1, sp)
    for f, sp in [(256, .75), (256, .72)]: x = res_block(x, f, 2 if f != x.shape[-1] else 1, sp)
    x = Dropout(0.4)(GlobalAveragePooling1D()(x))
    out = Dense(n_classes, activation="softmax")(x)
    m = Model(inp, out)
    m.compile(optimizer=Adam(3e-4), loss=CategoricalCrossentropy(label_smoothing=0.1), metrics=["accuracy"])
    return m

def build_vgg16(input_len, n_classes):
    def conv_block(x, f, n_convs):
        for _ in range(n_convs):
            x = ReLU()(BatchNormalization()(Conv1D(f, 3, padding="same", use_bias=False)(x)))
        return MaxPooling1D(2, padding="same")(x)
    inp = Input((input_len, 1))
    x = conv_block(inp, 32, 2)
    x = conv_block(x,   64, 2)
    x = conv_block(x,  128, 3)
    x = conv_block(x,  256, 3)
    x = conv_block(x,  256, 3)
    x = Dropout(0.4)(GlobalAveragePooling1D()(x))
    x = Dropout(0.4)(ReLU()(Dense(256)(x)))
    out = Dense(n_classes, activation="softmax")(x)
    m = Model(inp, out)
    m.compile(optimizer=Adam(3e-4), loss=CategoricalCrossentropy(label_smoothing=0.1), metrics=["accuracy"])
    return m

def inception_module(x, f1, f3r, f3, f5r, f5, fp):
    b1 = Conv1D(f1, 1, padding="same", activation="relu")(x)
    b3 = Conv1D(f3, 3, padding="same", activation="relu")(Conv1D(f3r, 1, padding="same", activation="relu")(x))
    b5 = Conv1D(f5, 5, padding="same", activation="relu")(Conv1D(f5r, 1, padding="same", activation="relu")(x))
    bp = Conv1D(fp, 1, padding="same", activation="relu")(MaxPooling1D(3, strides=1, padding="same")(x))
    return Concatenate()([b1, b3, b5, bp])

def build_inception(input_len, n_classes):
    inp = Input((input_len, 1))
    x = MaxPooling1D(3, strides=2, padding="same")(
            ReLU()(BatchNormalization()(Conv1D(32, 7, strides=2, padding="same")(inp))))
    x = inception_module(x, 16, 24, 32, 4,  8,  8)
    x = inception_module(x, 32, 32, 48, 8, 24, 16)
    x = MaxPooling1D(3, strides=2, padding="same")(x)
    x = inception_module(x, 48, 24, 52, 8, 24, 16)
    x = Dropout(0.4)(GlobalAveragePooling1D()(x))
    out = Dense(n_classes, activation="softmax")(x)
    m = Model(inp, out)
    m.compile(optimizer=Adam(3e-4), loss=CategoricalCrossentropy(label_smoothing=0.1), metrics=["accuracy"])
    return m

BUILDERS = {"resnet18": build_resnet18, "vgg16": build_vgg16, "inception": build_inception}

def run_deep_original(model_name, Xtr, ytr, Xva, yva, Xte, yte, seed, n_classes):
    tf.random.set_seed(seed)
    sc = StandardScaler()
    Xtr_d = sc.fit_transform(np.asarray(Xtr.todense() if hasattr(Xtr, "todense") else Xtr))[..., None]
    Xva_d = sc.transform(np.asarray(Xva.todense() if hasattr(Xva, "todense") else Xva))[..., None]
    Xte_d = sc.transform(np.asarray(Xte.todense() if hasattr(Xte, "todense") else Xte))[..., None]
    cw = dict(enumerate(compute_class_weight("balanced", classes=np.unique(ytr), y=ytr)))
    model = BUILDERS[model_name](Xtr_d.shape[1], n_classes)
    model.fit(Xtr_d, to_categorical(ytr, n_classes),
              validation_data=(Xva_d, to_categorical(yva, n_classes)),
              epochs=EPOCHS, batch_size=BATCH, class_weight=cw, shuffle=True, verbose=0,
              callbacks=[EarlyStopping("val_loss", patience=15, restore_best_weights=True),
                         ReduceLROnPlateau("val_loss", factor=0.5, patience=6, min_lr=1e-6)])
    prob = model.predict(Xte_d, verbose=0)
    pred = prob.argmax(1)
    f1 = f1_score(yte, pred, average="macro"); bal = balanced_accuracy_score(yte, pred)
    tf.keras.backend.clear_session()
    return f1, bal


### 6. Resume support

In [ ]:

RAW_PATH = f"{OUT}/ambig_ambiguity_all5_raw.csv"
if os.path.exists(RAW_PATH):
    existing = pd.read_csv(RAW_PATH)
    done_keys = set(zip(existing["model"], existing["length"].astype(str), existing["ambiguity_rate"], existing["seed"]))
    rows = existing.to_dict("records")
    print(f"Resuming: {len(existing)} existing rows, {len(done_keys)} combos already done.")
else:
    rows = []
    done_keys = set()
    print("No existing results -- starting fresh.")


### 7. Main sweep: (ambiguity rate x fragment length x seed x model), all five models

In [ ]:

for seed in SEEDS:
    tr, va, te = split_grouped(X_all, y_all, grp_all, seed)
    ytr, yva, yte = y_all[tr], y_all[va], y_all[te]
    rng = np.random.default_rng(seed)

    for L in LENGTHS:
        tag = L if L else "full"
        seq_tr = [fragment(X_all[i], L, rng) for i in tr]
        seq_va = [fragment(X_all[i], L, rng) for i in va]

        for a_rate in AMBIG_RATES:
            all_models = ["linear_svm", "random_forest"] + DEEP_MODELS
            if all((m, str(tag), a_rate, seed) in done_keys for m in all_models):
                continue

            seq_te = [inject_ambiguity(fragment(X_all[i], L, rng), a_rate, rng) for i in te]
            Xtr, Xva, Xte = fit_tfidf(seq_tr, seq_va, seq_te)

            for model_name, clf in [
                ("linear_svm", LinearSVC(class_weight="balanced", random_state=seed, max_iter=20000)),
                ("random_forest", RandomForestClassifier(n_estimators=300, class_weight="balanced",
                                                            random_state=seed, n_jobs=-1)),
            ]:
                if (model_name, str(tag), a_rate, seed) in done_keys:
                    continue
                clf.fit(Xtr, ytr)
                pred = clf.predict(Xte)
                rows.append(dict(model=model_name, length=tag, ambiguity_rate=a_rate, seed=seed,
                                  f1=f1_score(yte, pred, average="macro"),
                                  bal_acc=balanced_accuracy_score(yte, pred)))

            for model_name in DEEP_MODELS:
                if (model_name, str(tag), a_rate, seed) in done_keys:
                    continue
                f1, bal = run_deep_original(model_name, Xtr, ytr, Xva, yva, Xte, yte, seed, n_cls)
                rows.append(dict(model=model_name, length=tag, ambiguity_rate=a_rate, seed=seed,
                                  f1=f1, bal_acc=bal))

            pd.DataFrame(rows).to_csv(RAW_PATH, index=False)  # incremental save
            print(f"seed={seed} length={tag} ambiguity={a_rate} done ({len(rows)} rows so far)")


### 8. Aggregate and plot

In [ ]:

raw = pd.DataFrame(rows)
summ = raw.groupby(["model","length","ambiguity_rate"])[["f1","bal_acc"]].agg(["mean","std"]).round(4)
summ.columns = [f"{a}_{b}" for a,b in summ.columns]
summ = summ.reset_index()
summ.to_csv(f"{OUT}/ambig_ambiguity_all5_summary.csv", index=False)
print(summ)

length_order = [300, 500, 1000, "full"]
for model_name in raw["model"].unique():
    sub = summ[summ["model"] == model_name]
    pivot = sub.pivot(index="ambiguity_rate", columns="length", values="f1_mean")
    pivot = pivot[[c for c in length_order if c in pivot.columns]]
    fig, ax = plt.subplots(figsize=(6,4.5))
    im = ax.imshow(pivot.values, aspect="auto", cmap="viridis", vmin=0, vmax=1)
    ax.set_xticks(range(len(pivot.columns))); ax.set_xticklabels(pivot.columns)
    ax.set_yticks(range(len(pivot.index))); ax.set_yticklabels([f"{r*100:.1f}%" for r in pivot.index])
    ax.set_xlabel("fragment length (bp)"); ax.set_ylabel("injected ambiguity rate")
    ax.set_title(f"Ambig: macro-F1 surface (CD-HIT-97% split) — {model_name}")
    plt.colorbar(im, ax=ax, label="macro-F1")
    plt.tight_layout()
    plt.savefig(f"{OUT}/ambig_surface_{model_name}.png", dpi=200)
    plt.show()


### 9. Zip outputs

In [ ]:
!cd /kaggle/working && zip -r ambig_ambiguity_all5_outputs.zip ambig_ambiguity_all5_raw.csv ambig_ambiguity_all5_summary.csv ambig_surface_*.png